## Out-of-memory — executor vs. driver, and the fix patterns

OOM comes in two flavours, and the fix depends entirely on **which side** ran out.

**Executor OOM** — `java.lang.OutOfMemoryError: Java heap space` in a *task* log; tasks fail and the stage retries.

- **Partition too big** → raise `spark.sql.shuffle.partitions` so each fits.
- **Skew packing one task** → AQE skew-join.
- **`collect()` into a wide aggregation** → don't collect; write to a table.
- **UDF with high per-row memory** → batch with `pandas_udf`, or use built-ins.
- **Caching too much** → `unpersist` when done.
- *Last resort:* raise `spark.executor.memory`.

**Driver OOM** — `OutOfMemoryError` from the *driver*; the notebook hangs or the cluster restarts.

- **`collect()` / `toPandas()` on a big DataFrame** → never on more than a few MB.
- **Broadcasting a too-big table** → cap `spark.sql.autoBroadcastJoinThreshold`.
- **Massive event log / over-complex job** → simplify.
- *Last resort:* raise `spark.driver.memory`.

**The exam pattern:** *"an executor task fails with OOM after the shuffle."* → **raise shuffle partitions; AQE skew-join; broadcast the small side.** Bumping memory is the **last** lever, not the first. And the tell for *driver* OOM is always a `collect()` / `toPandas()` pulling too much back to one machine — the fix is *don't pull it back*, not a bigger driver.
